[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_1_Elimination_of_Constants.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs/Lab_1_Elimination_of_Constants.ipynb)


In [ ]:
# 🚀 ENVIRONMENT & PATH SETUP (Works in local JupyterLab, MyBinder, VS Code, and Google Colab)
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
except NameError:
    pass

# Automatically locate the repository root (containing labs/ and rover_path_finder/)
# regardless of whether JupyterLab sets cwd to the repo root, labs/, or solutions/labs/:
_cwd = os.path.abspath(os.getcwd())
for _candidate in [_cwd, os.path.abspath(os.path.join(_cwd, '..')), os.path.abspath(os.path.join(_cwd, '..', '..'))]:
    if os.path.isdir(os.path.join(_candidate, 'labs', 'tests')) and os.path.isdir(os.path.join(_candidate, 'rover_path_finder')):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break
print('[OK] Project root added to sys.path!')


# 📡 Lab 1: Origin of PDEs, Eliminating Arbitrary Constants, and Terrain Slope Maps
### *Mission Control: Reading Martian Topography & Local Slope Sensors*
**Course:** Applied Partial Differential Equations & Planetary Robotics
**Lecture Notes Reference:** Chapter 1, Sections 1.1 & 1.2 — **Notes pp. 1–10** *(PDF pp. 2–11)*

---
### 🗺️ Where You Are in the Rover Navigation Pipeline
```
📍 [Lab 1: Terrain Slopes (p, q)] ──► [Lab 2: Streamline Paths] ──► [Lab 3: Safety Buffers] ──► [Lab 4–5: Laplace Theory] ──► [Lab 6: Complete Rover Planner]
```

---
### 🧭 The Rosetta Stone: Connecting Lecture Notes to the Rover Engine

| 1. Lecture Notes Math (PDF pp. 2–11) | 2. Geometric & Physical Meaning | 3. Real Project Function (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $z = f(x, y)$ | Elevation (height) of the Martian surface at coordinates $(x, y)$ | `elev` array in [`terrain.py`](rover_path_finder/terrain.py) |
| $p = \frac{\partial z}{\partial x}, \quad q = \frac{\partial z}{\partial y}$ | Directional terrain slope along the East ($x$) and North ($y$) axes | `gy, gx = np.gradient(elev)` in [`terrain.slope_map`](rover_path_finder/terrain.py) |
| $\|\nabla z\| = \sqrt{p^2 + q^2}$ | Total steepness (slope magnitude) used to flag unsafe hills/craters (`> 0.30`) | `np.hypot(gx, gy)` inside [`terrain.slope_map(elev)`](rover_path_finder/terrain.py) |
| Eliminating arbitrary constants $(a, b)$ | Extracting a **universal slope law** $f(x, y, z, p, q) = 0$ independent of crater location $(a, b)$ | Evaluating the PDE residual `(P**2 + Q**2) - 4*Z` across the grid |

---
## 1. Intuition First: Why Do We Eliminate Arbitrary Constants?

Suppose a satellite scans a field of circular craters on Mars. Every crater has the same paraboloid bowl shape, but each sits at a different center $(a, b)$:
$$z(x, y) = (x - a)^2 + (y - b)^2$$

When the rover drives across this terrain, its onboard sensors measure its local height $z$ and the tilt of its wheels in the $x$ and $y$ directions:
$$p = \frac{\partial z}{\partial x}, \qquad q = \frac{\partial z}{\partial y}$$

The rover does **not** know the coordinates $(a, b)$ of every crater center in advance. By **eliminating the arbitrary constants $a$ and $b$** using partial differentiation, we obtain a single **Partial Differential Equation (PDE)** that relates the slope $(p, q)$ directly to the elevation $z$ at every point on every crater, regardless of where $(a, b)$ is located!


In [ ]:
import os, sys
for _p in ['.', '..', '../..']:
    _ap = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_ap, 'labs', 'tests')) and _ap not in sys.path:
        sys.path.insert(0, _ap)
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab1

%matplotlib inline
print("Lab 1 Environment Initialized! Ready for Terrain Slope Analysis.")


---
## 2. The Mathematical Recipe & Complete Lecture Notes Coverage (PDF pp. 2–11)

Before eliminating constants, let's master the **foundational definitions and classifications** from **Section 1.1 & Section 1.2** of your lecture notes (**Notes pp. 1–10 / PDF pp. 2–11**):

---
### Part A: Monge's Standard Notations, Order, and Degree (PDF pp. 2–4)

In this course, your lecture notes use **Monge's shorthand notations** for the partial derivatives of $z = f(x, y)$:
$$\boxed{p = \frac{\partial z}{\partial x}, \qquad q = \frac{\partial z}{\partial y}, \qquad r = \frac{\partial^2 z}{\partial x^2}, \qquad s = \frac{\partial^2 z}{\partial x \partial y}, \qquad t = \frac{\partial^2 z}{\partial y^2}}$$

* **Order of a PDE:** The order of the **highest partial derivative** appearing in the equation.
* **Degree of a PDE:** The **exponent (power) of that highest-order derivative**, **AFTER clearing all radicals (roots) and fractional powers** of the derivatives!

> ⚠️ **Exam Trap Alert (Notes p. 2, Equation (5) / PDF p. 3):**
> Look at Equation (5) in your lecture notes:
> $$\frac{\partial^2 z}{\partial x^2} = \sqrt{1 + \left(\frac{\partial z}{\partial y}\right)^2}$$
> * What is the **order**? The highest derivative is $\frac{\partial^2 z}{\partial x^2}$, which is **2nd order**.
> * What is the **degree**? Do **not** say 1! Because of the square root $\sqrt{\cdots}$ over a derivative, we must **square both sides first** to make the equation a polynomial in the derivatives:
>   $$\left(\frac{\partial^2 z}{\partial x^2}\right)^2 = 1 + \left(\frac{\partial z}{\partial y}\right)^2$$
>   Now the exponent on the highest derivative $\frac{\partial^2 z}{\partial x^2}$ is **$2$**, so this PDE has **Order = 2, Degree = 2**!

---
### Part B: The 4-Way Linearity Classification (Notes pp. 4–5 / PDF pp. 5–6)

How do we classify a PDE as **Linear**, **Semi-linear**, **Quasi-linear**, or **Fully Non-linear**? Your lecture notes define a clear hierarchy based on where $(x, y)$, $z$, and the highest derivatives appear:

| Classification (PDF pp. 5–6) | Highest-Order Derivatives | Coefficients of Highest Derivatives | Lower-Order Terms | Example from Lecture Notes & Rover Meaning |
| :--- | :--- | :--- | :--- | :--- |
| **1. Linear PDE** | Appear to the **1st power** only (no products) | Depend **only on $(x, y)$** (never on $z$ or derivatives) | Also strictly **linear** in $z$ and its lower derivatives | $y\frac{\partial z}{\partial x} + x\frac{\partial z}{\partial y} = z$ or Laplace's Eq. $\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ *(Lab 4–6 navigation potential)* |
| **2. Semi-linear PDE** | Appear to the **1st power** only | Depend **only on $(x, y)$** (never on $z$ or derivatives!) | Contain **non-linear** terms in $z$ or lower-order derivatives | $x^2\frac{\partial z}{\partial x} + y^2\frac{\partial z}{\partial y} = z^2$ *(the coefficients $x^2, y^2$ depend only on $x,y$, while the RHS has $z^2$)* |
| **3. Quasi-linear PDE** | Appear to the **1st power** only | May depend on **$z$ and/or lower-order derivatives** as well as $(x,y)$ | Can be non-linear in $z$ or lower derivatives | $(y - z)p + (z - x)q = x - y$ *(Lab 2 Lagrange streamlines where coefficients depend on $z$)* |
| **4. Non-linear PDE** | Raised to powers $>1$ (e.g., $p^2, q^2$) or multiplied together ($pq$) | Arbitrary | Arbitrary | $p^2 + q^2 = 1$ *(Lab 3 Eikonal obstacle safety buffer)* or $p^2 + q^2 = 4z$ *(Lab 1 crater slope law)* |

---
### Part C: Forming PDEs by Eliminating Arbitrary Constants — All 3 Situations (PDF pp. 7–10)

Given a surface family containing arbitrary constants, differentiating partially w.r.t. $x$ and $y$ produces two equations involving $p = \frac{\partial z}{\partial x}$ and $q = \frac{\partial z}{\partial y}$. Your lecture notes (**Notes pp. 6–8 / PDF pp. 7–9**) distinguish **three exact situations**:

#### 🔹 Situation 1: Fewer constants than independent variables ($1 < 2$) — *Notes p. 6 / PDF p. 7*
* **Rule:** Eliminating 1 constant using 2 partial derivatives gives **more than one** 1st-order PDE!
* **Lecture Notes Example ($z = \alpha x + y$):**
  * Differentiating w.r.t. $x$ gives $\frac{\partial z}{\partial x} = \alpha$, which leaves $\alpha$ in the equation unless we differentiate again ($\frac{\partial^2 z}{\partial x^2} = 0$ or $\frac{\partial^2 z}{\partial y\partial x} = 0$).
  * Differentiating w.r.t. $y$ directly eliminates $\alpha$ and yields the 1st-order PDE $\boxed{\frac{\partial z}{\partial y} = 1}$!

#### 🔹 Situation 2: Number of constants equals number of independent variables ($2 = 2$) — *Notes p. 7 / PDF p. 8*
* **Rule:** With 2 constants $(a, b)$ and 2 independent variables $(x, y)$, we have 3 equations (the original relation + the $x$-derivative + the $y$-derivative), which eliminates $(a, b)$ to yield a **unique 1st-order PDE**.
* **Worked Example 1 (Notes p. 8 / PDF pp. 9–10):** Eliminate $a, b$ from the family of planes:
  $$z = a x + b y + a^2 + b^2 \qquad \text{(1)}$$
  1. Differentiate w.r.t. $x$ (holding $y, a, b$ constant): $p = \frac{\partial z}{\partial x} = a \implies a = p \quad \text{(2)}$
  2. Differentiate w.r.t. $y$ (holding $x, a, b$ constant): $q = \frac{\partial z}{\partial y} = b \implies b = q \quad \text{(3)}$
  3. Substitute (2) and (3) into (1): $\boxed{z = p x + q y + p^2 + q^2}$ *(Clairaut form, which we solve in reverse in Lab 3!)*

#### 🔹 Situation 3: More constants than independent variables ($n > 2$) — *Notes p. 8 / PDF p. 9*
* **Rule:** First-order derivatives ($p, q$) only give 2 extra equations, which is not enough to eliminate 3 or more constants! You **must differentiate to 2nd order** ($r = z_{xx}, s = z_{xy}, t = z_{yy}$), giving a **higher-order PDE**.
* **Lecture Notes Example ($z = a x + b y + c x y$, with 3 constants $a, b, c$):**
  1. First derivatives: $p = \frac{\partial z}{\partial x} = a + c y$ and $q = \frac{\partial z}{\partial y} = b + c x$.
  2. Second derivatives immediately eliminate $a$ and $b$:
     $$\frac{\partial^2 z}{\partial x^2} = 0, \qquad \frac{\partial^2 z}{\partial y^2} = 0, \qquad \text{or} \quad \frac{\partial^2 z}{\partial x \partial y} = c \implies z = x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} - x y \frac{\partial^2 z}{\partial x \partial y}$$

> 📌 **PDF Page 9 Typo Alert (Notes p. 8):**
> At the bottom of page 8 in the lecture notes (PDF page 9), the notes write $\frac{\partial^2 y}{\partial x\partial y} = c$. The numerator $y$ is a printing typo in the PDF—the dependent variable is $z$, so it should be $\boxed{\frac{\partial^2 z}{\partial x\partial y} = c}$!

---
### ⚠️ Common Mistakes to Avoid
1. **Leaving $a$ or $b$ in your final answer:** A valid PDE obtained by elimination must contain only $(x, y, z, p, q)$—zero arbitrary constants may remain!
2. **Axis ordering in `np.gradient`:** In Python 2D arrays `Z[row, col]`, axis 0 is `y` (rows) and axis 1 is `x` (columns). Always call `Q, P = np.gradient(Z, dy, dx)` (just like `gy, gx = np.gradient(elev)` in [`terrain.py`](rover_path_finder/terrain.py)) so $P = \partial z/\partial x$ and $Q = \partial z/\partial y$ match the right axes!


---
## 3. 📝 Paper & Pen Challenge 1 (Notes p. 9, Example 2 / PDF p. 10)

**Problem:** Eliminate the arbitrary constants $a$ and $b$ from the family of circular paraboloid craters:
$$z = (x - a)^2 + (y - b)^2$$

**Work through these steps on paper:**
1. Differentiate partially w.r.t. $x$ to find $p = \frac{\partial z}{\partial x}$ in terms of $(x - a)$, then isolate $(x - a)$.
2. Differentiate partially w.r.t. $y$ to find $q = \frac{\partial z}{\partial y}$ in terms of $(y - b)$, then isolate $(y - b)$.
3. Substitute your expressions for $(x - a)$ and $(y - b)$ into the original equation $z = (x - a)^2 + (y - b)^2$ and clear fractions.

<details>
<summary>💡 <b>Hint 1 (Computing Partial Derivatives)</b></summary>

By the chain rule: $p = 2(x - a) \implies (x - a) = \frac{p}{2}$, and $q = 2(y - b) \implies (y - b) = \frac{q}{2}$.
</details>

<details>
<summary>💡 <b>Hint 2 (Substituting & Simplifying)</b></summary>

Substituting into $z = (x - a)^2 + (y - b)^2$ gives $z = \left(\frac{p}{2}\right)^2 + \left(\frac{q}{2}\right)^2 = \frac{p^2 + q^2}{4}$. Multiply both sides by $4$!
</details>


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ###
student_pde_1 = "p^2 + q^2 = 4z"
### END CODE HERE ###

# Run numerical verification test:
test_lab1.check_paper_challenge_1(student_pde_1)


---
## 4. 🚀 Rover Bridge Challenge (Elliptical Crater Basin)

Now suppose a Martian valley basin is modeled by a family of elliptical paraboloids with unknown curvatures $a$ and $b$:
$$z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2$$

**On your paper:**
1. Differentiate w.r.t. $x$ to find $p = \frac{\partial z}{\partial x}$ and solve for $a$ in terms of $p$ and $x$.
2. Differentiate w.r.t. $y$ to find $q = \frac{\partial z}{\partial y}$ and solve for $b$ in terms of $q$ and $y$.
3. Substitute $a$ and $b$ back into $z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2$ and multiply both sides by $2$.

<details>
<summary>💡 <b>Hint 1 (Isolating a and b)</b></summary>

Since $p = a x$ and $q = b y$, we have $a = \frac{p}{x}$ and $b = \frac{q}{y}$.
</details>

<details>
<summary>💡 <b>Hint 2 (Substituting into z)</b></summary>

Substituting gives $z = \frac{1}{2}\left(\frac{p}{x}\right)x^2 + \frac{1}{2}\left(\frac{q}{y}\right)y^2 = \frac{1}{2}(x p + y q)$. Multiply both sides by 2!
</details>


In [ ]:
# GRADED CELL: Rover Bridge Challenge
### START CODE HERE ###
rover_basin_pde = "x*p + y*q = 2z"
### END CODE HERE ###

# Run numerical verification test:
test_lab1.check_paper_challenge_2(rover_basin_pde)


---
## 5. 💻 Python Implementation: Verifying the Crater PDE on a Grid

From Challenge 1, every paraboloid crater $z = (x - a)^2 + (y - b)^2$ satisfies $p^2 + q^2 = 4z$, meaning the **PDE residual**:
$$\text{residual}(x, y) = (p^2 + q^2) - 4z$$
must equal zero everywhere on the surface! Implement `verify_crater_pde(Z, P, Q)` below using vectorized NumPy operations.


In [ ]:
# GRADED FUNCTION: verify_crater_pde
def verify_crater_pde(Z: np.ndarray, P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    # Computes the PDE residual: (p^2 + q^2) - 4*z
    ### START CODE HERE ###
    residual = (P**2 + Q**2) - 4.0 * Z
    ### END CODE HERE ###
    return residual

# Run automated test:
test_lab1.test_eliminate_constants_fn(verify_crater_pde)


---
## 6. 🎨 Visualizing the Physics & Connecting to `rover_path_finder/terrain.py`

Run the cell below to inspect three views:
1. **Left:** A 3D paraboloid crater $z = (x-a)^2 + (y-b)^2$.
2. **Center:** The 2D slope vector field $(p, q) = (\partial z/\partial x, \partial z/\partial y)$ computed with the exact axis ordering `Q, P = np.gradient(Z, dy, dx)`.
3. **Right:** Calling [`terrain.slope_map(elev)`](rover_path_finder/terrain.py) from our actual rover codebase on synthetic Martian terrain to detect steep hazard regions (`|\nabla z| > 0.30`).


In [ ]:
from rover_path_finder import terrain

x = np.linspace(-3, 3, 41)
y = np.linspace(-3, 3, 41)
X, Y = np.meshgrid(x, y)
Z = (X - 0.5)**2 + (Y + 0.5)**2  # Center at (a=0.5, b=-0.5)

dx = x[1] - x[0]
dy = y[1] - y[0]
# Axis 0 is y (rows), Axis 1 is x (cols):
Q, P = np.gradient(Z, dy, dx)

# Real project terrain & slope hazard map from rover_path_finder/terrain.py:
t_data = terrain.make_terrain(N=100, seed=42)
elev, raw_hazard = t_data["h"], t_data["raw"]
slope_mag = terrain.slope_map(elev, cell=1.0)

fig = plt.figure(figsize=(16, 4.8))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='copper', alpha=0.88)
ax1.set_title("1. Analytical Crater $z = (x-a)^2 + (y-b)^2$", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")

ax2 = fig.add_subplot(1, 3, 2)
ax2.contour(X, Y, Z, levels=14, cmap='copper')
ax2.quiver(X[::3, ::3], Y[::3, ::3], P[::3, ::3], Q[::3, ::3], color='crimson')
ax2.set_title(r"2. Slope Vectors $(p, q) = (\partial z/\partial x, \partial z/\partial y)$", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_aspect("equal")

ax3 = fig.add_subplot(1, 3, 3)
im = ax3.imshow(slope_mag, cmap='inferno', origin='upper')
ax3.contour(raw_hazard, levels=[0.5], colors='cyan', linewidths=1.2)
plt.colorbar(im, ax=ax3, label=r"$\|\nabla z\| = \sqrt{p^2 + q^2}$")
ax3.set_title("3. Real Engine: terrain.slope_map(elev) & Hazards", fontsize=11)
ax3.set_xlabel("x (pixels)")
ax3.set_ylabel("y (pixels)")

plt.tight_layout()
plt.show()

interior_err = np.abs(verify_crater_pde(Z, P, Q)[1:-1, 1:-1]).max()
print(f"Max interior residual |(p^2 + q^2) - 4z|: {interior_err:.4f}")
print("Lab 1 Takeaways:")
print("  1) Eliminating (a, b) produces a universal PDE relating height z to local slopes (p, q).")
print("  2) In rover_path_finder/terrain.py, slope_map(elev) computes sqrt(p^2 + q^2) to flag steep craters.")
